In [1]:
import numpy as np
from collections import Counter

print("========== AUTOREGRESSIVE CAUSAL LANGUAGE MODEL ==========\n")

# --------------------------------------------------
# Step 1 : Create Corpus
# --------------------------------------------------

corpus = [
    "study",
    "student",
    "students",
    "studying",
    "teach",
    "teacher",
    "teaching",
    "taught",
    "learn",
    "learner",
    "learning",
    "learned"
]

print("Corpus:\n")
for word in corpus:
    print(word)

# --------------------------------------------------
# Step 2 : Convert Words into Characters
# --------------------------------------------------

tokens = []

for word in corpus:
    for ch in word:
        tokens.append(ch)
    tokens.append("</w>")

print("\nTokens:\n")
print(tokens)

# --------------------------------------------------
# Step 3 : Build Vocabulary
# --------------------------------------------------

vocabulary = []

for token in tokens:
    if token not in vocabulary:
        vocabulary.append(token)

print("\nVocabulary:\n")
print(vocabulary)

# --------------------------------------------------
# Step 4 : Token to ID
# --------------------------------------------------

token_to_id = {}
id_to_token = {}

for i, token in enumerate(vocabulary):
    token_to_id[token] = i
    id_to_token[i] = token

print("\nToken IDs:\n")

for token in token_to_id:
    print(token, "->", token_to_id[token])

# --------------------------------------------------
# Step 5 : Convert Tokens to IDs
# --------------------------------------------------

token_ids = []

for token in tokens:
    token_ids.append(token_to_id[token])

print("\nToken ID Sequence:\n")
print(token_ids)

# --------------------------------------------------
# Step 6 : Create Input & Target
# --------------------------------------------------

inputs = np.array(token_ids[:-1])
targets = np.array(token_ids[1:])

print("\nInputs:\n")
print(inputs)

print("\nTargets:\n")
print(targets)

========== AUTOREGRESSIVE CAUSAL LANGUAGE MODEL ==========

Corpus:

study
student
students
studying
teach
teacher
teaching
taught
learn
learner
learning
learned

Tokens:

['s', 't', 'u', 'd', 'y', '</w>', 's', 't', 'u', 'd', 'e', 'n', 't', '</w>', 's', 't', 'u', 'd', 'e', 'n', 't', 's', '</w>', 's', 't', 'u', 'd', 'y', 'i', 'n', 'g', '</w>', 't', 'e', 'a', 'c', 'h', '</w>', 't', 'e', 'a', 'c', 'h', 'e', 'r', '</w>', 't', 'e', 'a', 'c', 'h', 'i', 'n', 'g', '</w>', 't', 'a', 'u', 'g', 'h', 't', '</w>', 'l', 'e', 'a', 'r', 'n', '</w>', 'l', 'e', 'a', 'r', 'n', 'e', 'r', '</w>', 'l', 'e', 'a', 'r', 'n', 'i', 'n', 'g', '</w>', 'l', 'e', 'a', 'r', 'n', 'e', 'd', '</w>']

Vocabulary:

['s', 't', 'u', 'd', 'y', '</w>', 'e', 'n', 'i', 'g', 'a', 'c', 'h', 'r', 'l']

Token IDs:

s -> 0
t -> 1
u -> 2
d -> 3
y -> 4
</w> -> 5
e -> 6
n -> 7
i -> 8
g -> 9
a -> 10
c -> 11
h -> 12
r -> 13
l -> 14

Token ID Sequence:

[0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 6, 7, 1, 5, 0, 1, 2, 3, 6, 7, 1, 0, 5, 0, 1, 2, 3, 4, 8

In [2]:
# --------------------------------------------------
# Step 7 : Create Embedding Matrix
# --------------------------------------------------

embedding_dimension = 8
vocab_size = len(vocabulary)

embedding_matrix = np.random.randn(
    vocab_size,
    embedding_dimension
)

print("\nEmbedding Matrix Shape:")
print(embedding_matrix.shape)

# --------------------------------------------------
# Step 8 : Convert Inputs to Embeddings
# --------------------------------------------------

embedded_inputs = []

for value in inputs:
    embedded_inputs.append(embedding_matrix[value])

embedded_inputs = np.array(embedded_inputs)

print("\nEmbedded Input Shape:")
print(embedded_inputs.shape)

# --------------------------------------------------
# Step 9 : Positional Encoding
# --------------------------------------------------

sequence_length = len(inputs)

position_encoding = np.zeros(
    (sequence_length, embedding_dimension)
)

for pos in range(sequence_length):

    for i in range(embedding_dimension):

        angle = pos / np.power(
            10000,
            (2 * (i // 2)) / embedding_dimension
        )

        if i % 2 == 0:
            position_encoding[pos][i] = np.sin(angle)
        else:
            position_encoding[pos][i] = np.cos(angle)

print("\nPositional Encoding Shape:")
print(position_encoding.shape)

# Add Position Encoding
embedded_inputs = embedded_inputs + position_encoding

print("\nEmbedding + Position Encoding Added")

# --------------------------------------------------
# Step 10 : Create Weight Matrices
# --------------------------------------------------

W_query = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

W_key = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

W_value = np.random.randn(
    embedding_dimension,
    embedding_dimension
)

print("\nWeight Matrices Created")

# --------------------------------------------------
# Step 11 : Calculate Q, K and V
# --------------------------------------------------

Q = np.dot(embedded_inputs, W_query)
K = np.dot(embedded_inputs, W_key)
V = np.dot(embedded_inputs, W_value)

print("\nQuery Shape :", Q.shape)
print("Key Shape   :", K.shape)
print("Value Shape :", V.shape)

# --------------------------------------------------
# Step 12 : Attention Scores
# --------------------------------------------------

scores = np.dot(Q, K.T)
scores = scores / np.sqrt(embedding_dimension)

print("\nAttention Score Shape:")
print(scores.shape)

# --------------------------------------------------
# Step 13 : Apply Causal Mask
# --------------------------------------------------

mask = np.zeros((sequence_length, sequence_length))

for i in range(sequence_length):
    for j in range(sequence_length):
        if j > i:
            mask[i][j] = -1000000000

scores = scores + mask

print("\nCausal Mask Applied Successfully")


Embedding Matrix Shape:
(15, 8)

Embedded Input Shape:
(92, 8)

Positional Encoding Shape:
(92, 8)

Embedding + Position Encoding Added

Weight Matrices Created

Query Shape : (92, 8)
Key Shape   : (92, 8)
Value Shape : (92, 8)

Attention Score Shape:
(92, 92)

Causal Mask Applied Successfully


In [3]:
# --------------------------------------------------
# Step 14 : Softmax Function
# --------------------------------------------------

def softmax(matrix):

    output = np.zeros(matrix.shape)

    for i in range(matrix.shape[0]):

        row = matrix[i]

        row = row - np.max(row)

        exp = np.exp(row)

        output[i] = exp / np.sum(exp)

    return output

# --------------------------------------------------
# Step 15 : Attention Weights
# --------------------------------------------------

attention_weights = softmax(scores)

print("\nAttention Weight Shape:")
print(attention_weights.shape)

# --------------------------------------------------
# Step 16 : Attention Output
# --------------------------------------------------

attention_output = np.dot(
    attention_weights,
    V
)

print("\nAttention Output Shape:")
print(attention_output.shape)

# --------------------------------------------------
# Step 17 : Output Layer
# --------------------------------------------------

W_output = np.random.randn(
    embedding_dimension,
    vocab_size
)

bias = np.zeros(vocab_size)

logits = np.dot(
    attention_output,
    W_output
) + bias

print("\nLogits Shape:")
print(logits.shape)

# --------------------------------------------------
# Step 18 : Probability
# --------------------------------------------------

probabilities = softmax(logits)

print("\nProbability Shape:")
print(probabilities.shape)

print("\nFirst Probability Vector:")
print(probabilities[0])

# --------------------------------------------------
# Step 19 : Predict Next Token
# --------------------------------------------------

prediction = np.argmax(probabilities[0])

print("\nPredicted Next Token:")
print(id_to_token[prediction])

# --------------------------------------------------
# Step 20 : Cross Entropy Loss
# --------------------------------------------------

def cross_entropy_loss(probabilities, targets):

    loss = 0

    for i in range(len(targets)):

        p = probabilities[i][targets[i]]

        if p < 1e-10:
            p = 1e-10

        loss -= np.log(p)

    loss = loss / len(targets)

    return loss

loss = cross_entropy_loss(
    probabilities,
    targets
)

print("\nInitial Loss:")
print(loss)


Attention Weight Shape:
(92, 92)

Attention Output Shape:
(92, 8)

Logits Shape:
(92, 15)

Probability Shape:
(92, 15)

First Probability Vector:
[1.23282261e-02 3.25597668e-08 8.99367518e-06 2.78231896e-01
 5.20385980e-07 6.20047394e-05 1.36525239e-02 5.77057057e-04
 2.05535144e-05 4.06808663e-03 5.58969582e-01 1.31520414e-01
 2.28725067e-04 3.25444613e-04 5.94051915e-06]

Predicted Next Token:
a

Initial Loss:
9.614114920610705


In [4]:
# --------------------------------------------------
# Step 21 : Training Loop
# --------------------------------------------------

learning_rate = 0.01
epochs = 100

for epoch in range(epochs):

    # Forward Pass
    Q = np.dot(embedded_inputs, W_query)
    K = np.dot(embedded_inputs, W_key)
    V = np.dot(embedded_inputs, W_value)

    scores = np.dot(Q, K.T)
    scores = scores / np.sqrt(embedding_dimension)
    scores = scores + mask

    attention_weights = softmax(scores)

    attention_output = np.dot(attention_weights, V)

    logits = np.dot(attention_output, W_output) + bias

    probabilities = softmax(logits)

    loss = cross_entropy_loss(probabilities, targets)

    # Gradient
    gradient = probabilities.copy()

    for i in range(len(targets)):
        gradient[i][targets[i]] -= 1

    gradient = gradient / len(targets)

    # Update Output Layer
    dW = np.dot(attention_output.T, gradient)
    db = np.sum(gradient, axis=0)

    W_output = W_output - learning_rate * dW
    bias = bias - learning_rate * db

    if (epoch + 1) % 10 == 0:
        print("Epoch", epoch + 1, "Loss =", loss)

# --------------------------------------------------
# Step 22 : Generate Next Token
# --------------------------------------------------

def generate_next_token(current_token):

    token_id = token_to_id[current_token]

    embedding = embedding_matrix[token_id]
    embedding = embedding.reshape(1, embedding_dimension)

    Q = np.dot(embedding, W_query)
    K = np.dot(embedding, W_key)
    V = np.dot(embedding, W_value)

    score = np.dot(Q, K.T)

    weight = softmax(score)

    context = np.dot(weight, V)

    logits = np.dot(context, W_output) + bias

    probability = softmax(logits)

    next_id = np.argmax(probability)

    return id_to_token[next_id]

# --------------------------------------------------
# Step 23 : Text Generation
# --------------------------------------------------

print("\nTEXT GENERATION")

start_token = "s"      # Change to t or l if needed

generated = [start_token]

current = start_token

for i in range(10):

    next_token = generate_next_token(current)

    generated.append(next_token)

    current = next_token

    if next_token == "</w>":
        break

print("\nGenerated Tokens:")
print(generated)

# --------------------------------------------------
# Step 24 : Convert Tokens to Word
# --------------------------------------------------

word = ""

for token in generated:

    if token != "</w>":
        word += token

print("\nGenerated Word:")
print(word)

print("\n========== AUTOREGRESSIVE MODEL COMPLETED ==========")

Epoch 10 Loss = 9.13640388639692
Epoch 20 Loss = 8.679485144615473
Epoch 30 Loss = 8.262455646214383
Epoch 40 Loss = 7.8888685723759275
Epoch 50 Loss = 7.553582339015412
Epoch 60 Loss = 7.256589555018993
Epoch 70 Loss = 6.9986914420084005
Epoch 80 Loss = 6.7765477574999595
Epoch 90 Loss = 6.583183033471105
Epoch 100 Loss = 6.412387278920853

TEXT GENERATION

Generated Tokens:
['s', '</w>']

Generated Word:
s

========== AUTOREGRESSIVE MODEL COMPLETED ==========
